# Single-turn coding control (Qwen3-8B, PROXY axis) — DebugBench vs. the frozen value axis

**Preliminary / not citable as-is.** Uses `value_axis_proxy.npy` — the
**dev/plumbing-test axis** (`stage1/config/presets/dev.yaml`): built from
cheap local-Qwen-generated syntactic ICRL data, gated at a looser 0.75
threshold, no surviving manifest (no recorded AUROC or confirmed primary
layer — using layer 21 from the dev preset's `gate_layers` as a working
assumption). This is **not** the faithful 8B axis (that one currently has a
bug — saved as 128-dim instead of 4096 — and needs to be rebuilt in Stage 1
before a trustworthy 8B result exists).

**Why run this anyway:** the 32B run (`single_turn_control_colab.ipynb`)
found a real, statistically significant signal at layer 49, but with
**inconsistent sign** between the mean-over-code-tokens and final-token
readouts — strongest and most inverted for the `shuffled` corruption. A
per-token trace showed the projection ramps up positionally across the code
span in both correct and corrupted code, and shuffling scrambles *where*
that ramp lands, which plausibly explains the mean/final disagreement more
than it reflects code correctness per se.

**This run changes two things at once to probe that:**
1. **Model scale**: Qwen3-8B instead of 32B.
2. **Thinking mode: OFF** (`enable_thinking=False`), unlike the 32B run's
   `True`. This matches Jiang et al.'s own Fig. 4 methodology (their code
   correlation experiment used thinking OFF) *and* how this proxy axis
   itself was built (`enable_thinking: false` in `stage1/config/defaults.yaml`).
   If the mean/final disagreement is driven by thinking-mode, it should
   shrink or disappear here; if it's driven by the code's positional
   structure regardless of thinking mode, it should persist.

**Runtime:** Qwen3-8B bf16 is ~16GB — far lighter than the 32B run. An
**L4 (24GB)** or **A100 (40/80GB)** is comfortable; a T4 (16GB) is too tight
(model alone nearly fills it, no room for activations). Uses far fewer
compute units than the 32B notebook.

**Outputs (on Drive):** `problems_manifest.json`, `projections.parquet`,
`report/report.json`, `report/layer_sweep.png`, all under
`single_turn_control_8b_proxy/` (kept separate from the 32B run's folder).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need a GPU'
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(name, 'VRAM GB:', vram)
assert vram >= 20, (
    f'Got {vram} GB — Qwen3-8B bf16 needs headroom beyond its ~16GB weights. '
    'Runtime -> Change runtime type -> L4 or A100 (T4 16GB is too tight).'
)

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}
!pip install -q -e stage1 -e stage2 -e single_turn_control

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/single_turn_control_8b_proxy')
OUT_DIR = DRIVE_ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
PRIMARY_LAYER = 21          # dev preset's gate_layers[0] — NOT a confirmed-best layer (no manifest)
MODEL = 'Qwen/Qwen3-8B'
N_LAYERS = 36
N_PROBLEMS = 150
SEED = 42
ENABLE_THINKING = False     # deliberate: matches Jiang et al. Fig 4 + this axis's own construction mode

print('PRIMARY_LAYER', PRIMARY_LAYER, '(working assumption, no manifest to confirm this)')
print('MODEL', MODEL)
print('ENABLE_THINKING', ENABLE_THINKING)

## Upload input (one dialog)

Just `value_axis_proxy.npy` — there's no manifest for this axis, so
`analyze.py` is told the primary layer explicitly via `--primary-layer`
instead of reading it from a manifest file.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
import numpy as np

REPO = Path('/content/failure_prediction_research')
AXIS_DIR = REPO / 'stage1' / 'data'
AXIS_DIR.mkdir(parents=True, exist_ok=True)

def _take_one(upload_dict, dest: Path, *, expect_suffix: str | None = None):
    assert len(upload_dict) == 1, f'Upload exactly one file, got {list(upload_dict)}'
    name = next(iter(upload_dict))
    src = Path(name)
    if expect_suffix is not None:
        assert src.suffix.lower() == expect_suffix.lower(), (
            f'Expected {expect_suffix}, got {src.name}'
        )
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        dest.unlink()
    shutil.move(str(src), dest)
    print('saved ->', dest)
    return dest

print('Upload value_axis_proxy.npy')
AXIS = _take_one(files.upload(), AXIS_DIR / 'value_axis_proxy.npy', expect_suffix='.npy')
axis = np.load(AXIS)
print('axis shape', axis.shape, '(expect 36 x 4096)')
assert axis.shape == (36, 4096), axis.shape

## Prepare (CPU) — DebugBench -> corrupted variants -> rendered prompts

Same 150-problem seeded sample as the 32B run (same `SEED=42`), so results
are comparable problem-for-problem — but rendered with
`enable_thinking=False` this time.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
MANIFEST_PATH = OUT_DIR / 'problems_manifest.json'

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.prepare',
    '--n-problems', str(N_PROBLEMS),
    '--seed', str(SEED),
    '--model', MODEL,
    '--output', str(MANIFEST_PATH),
]
if not ENABLE_THINKING:
    cmd.append('--no-enable-thinking')
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('prepare exit:', rc, flush=True)
assert rc == 0 and MANIFEST_PATH.exists(), MANIFEST_PATH
print('problems_manifest ->', MANIFEST_PATH)

## Project (GPU)

All 36 layers, one forward pass per (problem, variant). Checkpoints after
each problem, mirrored to Drive every 10. Re-run to resume.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
AXIS = REPO / 'stage1' / 'data' / 'value_axis_proxy.npy'

LOCAL_OUT = Path('/content/single_turn_control_8b_ckpt')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
PROJ = LOCAL_OUT / 'projections.parquet'
DRIVE_PROJ = OUT_DIR / 'projections.parquet'
MIRROR_EVERY = 10

if DRIVE_PROJ.exists() and (not PROJ.exists() or DRIVE_PROJ.stat().st_size > PROJ.stat().st_size):
    print(f'copying {DRIVE_PROJ} -> {PROJ} ...', flush=True)
    shutil.copy2(DRIVE_PROJ, PROJ)

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.run_control',
    '--problems-manifest', str(MANIFEST_PATH),
    '--axis-path', str(AXIS),
    '--model', MODEL,
    '--n-layers', str(N_LAYERS),
    '--output', str(PROJ),
    '--mirror-output', str(DRIVE_PROJ),
    '--mirror-every', str(MIRROR_EVERY),
]
print('CMD:', ' '.join(cmd), flush=True)
print('Quiet while loading Qwen3-8B is normal (much faster than 32B)...', flush=True)

proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('run_control exit:', rc, flush=True)
assert rc == 0 and PROJ.exists(), PROJ
print('projections ->', PROJ, 'size MB', round(PROJ.stat().st_size / 1e6, 2))
print('Drive mirror:', DRIVE_PROJ)

## Analyze (CPU)

`--primary-layer` passed explicitly (21) since there's no manifest to read
it from.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
REPORT_DIR = OUT_DIR / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.analyze',
    '--projections', str(DRIVE_PROJ),
    '--output-dir', str(REPORT_DIR),
    '--primary-layer', str(PRIMARY_LAYER),
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('analyze exit:', rc, flush=True)
assert rc == 0
print('report ->', REPORT_DIR)

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

rep = OUT_DIR / 'report' / 'report.json'
if rep.exists():
    print(json.dumps(json.loads(rep.read_text()), indent=2)[:4000])

plot = OUT_DIR / 'report' / 'layer_sweep.png'
if plot.exists():
    display(Image(filename=str(plot)))
else:
    print('missing', plot)

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = OUT_DIR / 'single_turn_control_8b_proxy_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.json', '.png', '.parquet'}:
            z.write(p, p.relative_to(OUT_DIR).as_posix())
print('zip ->', zip_path)
files.download(str(zip_path))